# Semantic metric view
Run after notebook 01. Select the same catalog. If metric view creation fails due to feature availability,
use the KPI fallback already created by 01 and record that limitation. Do not suppress other SQL errors.


In [ ]:
# Set this to an existing writable catalog from Catalog Explorer.
CATALOG = "workspace"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.conf.set("spark.sql.session.timeZone", "UTC")
for schema in ["cj_bronze", "cj_silver", "cj_gold", "cj_graph", "cj_semantic"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema}`")
print("Catalog:", CATALOG)


In [ ]:
spark.sql("CREATE OR REPLACE VIEW cj_semantic.communication_metrics\nWITH METRICS LANGUAGE YAML AS $$\nversion: 1.1\nsource: __CATALOG__.cj_gold.communication_outcomes\nfilter: outcome <> 'PENDING'\ndimensions:\n  - name: campaign\n    expr: campaign_id\n  - name: region\n    expr: region\n  - name: activation_date\n    expr: DATE(activation_ts)\n  - name: snapshot_id\n    expr: snapshot_id\nmeasures:\n  - name: expected_communications\n    expr: COUNT(*)\n  - name: on_time_communications\n    expr: SUM(CASE WHEN outcome = 'ON_TIME' THEN 1 ELSE 0 END)\n  - name: late_communications\n    expr: SUM(CASE WHEN outcome = 'LATE' THEN 1 ELSE 0 END)\n  - name: unconfirmed_communications\n    expr: SUM(CASE WHEN outcome = 'NO_CONFIRMED_DELIVERY' THEN 1 ELSE 0 END)\n  - name: sla_breaches\n    expr: SUM(CASE WHEN outcome IN ('LATE', 'NO_CONFIRMED_DELIVERY') THEN 1 ELSE 0 END)\n  - name: sla_breach_rate\n    expr: SUM(CASE WHEN outcome IN ('LATE', 'NO_CONFIRMED_DELIVERY') THEN 1.0 ELSE 0.0 END) / NULLIF(COUNT(*), 0)\n  - name: affected_customers\n    expr: COUNT(DISTINCT CASE WHEN outcome IN ('LATE', 'NO_CONFIRMED_DELIVERY') THEN customer_id END)\n$$;\n".replace("__CATALOG__", CATALOG))

In [ ]:
display(spark.sql("""SELECT campaign, MEASURE(expected_communications) expected,
 MEASURE(sla_breaches) breaches, MEASURE(sla_breach_rate) breach_rate
 FROM cj_semantic.communication_metrics GROUP BY campaign"""))
# Expected: WELCOME, 10000, 500, 0.05
